# 🎬 WAN 2.1 (1.3B / 14B GGUF) trên Google Colab Free với ComfyUI
### Bộ công cụ 1-Click tối ưu hóa cho YouTube Faceless, Shorts, TikTok & Chống tràn RAM (OOM)

Notebook này thiết lập trọn gói môi trường để tạo video AI điện ảnh với mô hình **Wan 2.1** (Image-to-Video, Text-to-Video, First Frame - Last Frame):
- ⚡ **Tải Model tốc độ cao:** Dùng `aria2` & `huggingface_hub` siêu tốc (~1-2 phút cho bộ model).
- 🛡️ **Tự động vá lỗi hệ thống:** Đồng bộ PyTorch 2.6 cu124, patch `comfy-kitchen`, chống kẹt splash screen.
- 💾 **Google Drive Auto-Sync:** Tự động tạo Symlink lưu thẳng toàn bộ video đã render vào Drive cá nhân (`MyDrive/Wan21_Videos`).
- 🌐 **Cloudflare Tunnel (Legacy Frontend):** Mở Web UI tức thì trong 0.5s, không cần ngrok token.

## Bước 1: Cài đặt PyTorch, ComfyUI, Custom Nodes & Vá lỗi Type Schema

In [ ]:
#@title 1. Cài đặt Môi trường, ComfyUI & Tự động Vá lỗi { display-mode: "form" }
import os, glob, re, subprocess

print("🔍 1. Kiểm tra cấu hình GPU Tesla T4...")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

print("\n📦 2. Cài đặt aria2 & Đồng bộ PyTorch 2.6 cu124 chuẩn...")
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q -U xformers --index-url https://download.pytorch.org/whl/cu124
!pip install -q huggingface_hub

print("\n🚀 3. Clone ComfyUI Core & Cài đặt Custom Nodes...")
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt

%cd /content/ComfyUI/custom_nodes
for repo in [
    'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
    'https://github.com/city96/ComfyUI-GGUF.git',
    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git'
]:
    name = repo.split('/')[-1].replace('.git', '')
    if not os.path.exists(name):
        subprocess.run(f'git clone {repo}', shell=True, check=True)
        req = os.path.join(name, 'requirements.txt')
        if os.path.exists(req):
            subprocess.run(f'pip install -q -r {req}', shell=True, check=True)

print("\n🛠️ 4. Tự động vá lỗi comfy-kitchen type-hinting...")
!pip install --force-reinstall -q comfy-kitchen
for pyfile in glob.glob("/usr/local/lib/python3.12/dist-packages/comfy_kitchen/**/*.py", recursive=True):
    with open(pyfile, "r", encoding="utf-8") as f:
        s = f.read()
    s = s.replace("list[int]", "typing.List[int]").replace("list[bool]", "typing.List[bool]").replace("list[float]", "typing.List[float]")
    if "import typing" not in s:
        if "from __future__ import" in s:
            s = re.sub(r'(from __future__ import[^\n]+\n)', r'\1import typing\n', s, count=1)
        else:
            s = "import typing\n" + s
    with open(pyfile, "w", encoding="utf-8") as f:
        f.write(s)

print("\n✅ Hoàn tất cài đặt và vá lỗi hệ thống 100%!")

## Bước 2: Tải Models Wan 2.1 (T2V 1.3B & I2V 14B GGUF)

In [ ]:
#@title 2. Tải Trọng số Wan 2.1 (Tự động tải Core + I2V 14B GGUF + T2V 1.3B) { display-mode: "form" }
import os, subprocess
from huggingface_hub import snapshot_download

download_i2v_14B_GGUF = True #@param {type:"boolean"}
download_t2v_1_3B = True #@param {type:"boolean"}

%cd /content/ComfyUI

def download_aria(url, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    target_path = os.path.join(out_dir, filename)
    if not os.path.exists(target_path):
        print(f"⏳ Đang tải {filename}...")
        cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --allow-overwrite=true "{url}" -d "{out_dir}" -o "{filename}"'
        try:
            subprocess.run(cmd, shell=True, check=True)
            print(f"✅ Đã tải xong {filename}")
        except Exception:
            print(f"⚠️ Tải bằng wget dự phòng...")
            subprocess.run(f'wget -c "{url}" -O "{target_path}"', shell=True, check=True)
            print(f"✅ Đã tải xong {filename}")
    else:
        print(f"⚡ File {filename} đã có sẵn!")

print("📥 1. Tải các thành phần cốt lõi chung (Text Encoder FP8 Scaled, VAE, CLIP Vision)...")
# 1. Text Encoder UMT5 FP8 Scaled
download_aria(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "/content/ComfyUI/models/text_encoders",
    "umt5_xxl_fp8_e4m3fn_scaled.safetensors"
)
# 2. Wan 2.1 VAE
download_aria(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
    "/content/ComfyUI/models/vae",
    "wan_2.1_vae.safetensors"
)
# 3. CLIP Vision Model
download_aria(
    "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/clip_vision/clip_vision_h.safetensors",
    "/content/ComfyUI/models/clip_vision",
    "clip_vision_h.safetensors"
)

if download_i2v_14B_GGUF:
    print("\n📥 2. Tải mô hình Wan 2.1 I2V 14B GGUF Q4_K_M (Biến ảnh thành video siêu nét)...")
    snapshot_download(
        repo_id="city96/Wan2.1-I2V-14B-480P-gguf",
        allow_patterns="*Q4_K_M*",
        local_dir="/content/ComfyUI/models/unet"
    )
    print("✅ Đã tải xong Wan 2.1 I2V 14B GGUF!")

if download_t2v_1_3B:
    print("\n📥 3. Tải mô hình Wan 2.1 T2V 1.3B (Tạo video từ prompt nhanh nhẹ)...")
    download_aria(
        "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors",
        "/content/ComfyUI/models/diffusion_models",
        "wan2.1_t2v_1.3B_bf16.safetensors"
    )

print("\n🎉 TẤT CẢ MODEL ĐÃ ĐƯỢC NẠP HOÀN TẤT SẴN SÀNG!")

## Bước 3: Tự động kết nối Google Drive (Auto-Sync Output an toàn)

In [ ]:
#@title 3. (Tùy chọn) Kết nối Google Drive để lưu video vĩnh viễn { display-mode: "form" }
import os, shutil

save_to_drive = True #@param {type:"boolean"}

if save_to_drive:
    try:
        from google.colab import drive
        print("📂 Đang kết nối Google Drive...")
        drive.mount('/content/drive')
        drive_output = '/content/drive/MyDrive/Wan21_Videos'
        comfy_output = '/content/ComfyUI/output'
        os.makedirs(drive_output, exist_ok=True)
        
        if os.path.exists(comfy_output) and not os.path.islink(comfy_output):
            shutil.rmtree(comfy_output, ignore_errors=True)
        if not os.path.exists(comfy_output):
            os.symlink(drive_output, comfy_output)
            
        print(f"✅ Video tạo ra sẽ tự động lưu vĩnh viễn vào Google Drive: {drive_output}")
    except Exception as e:
        print(f"⚠️ Bỏ qua kết nối Drive do: {e}")
        print("ℹ️ Video vẫn được tạo và lưu bình thường tại /content/ComfyUI/output")
else:
    print("ℹ️ Video sẽ lưu tạm thời tại /content/ComfyUI/output")

## Bước 4: Khởi động ComfyUI (Legacy Frontend - Load trong 0.5s) & Mở Cloudflare

In [ ]:
#@title 4. KHỞI CHẠY COMFYUI & LẤY ĐƯỜNG LINK TRUY CẬP WEB { display-mode: "form" }
import subprocess, time, re

# 1. Cài đặt cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 2. Dừng tiến trình cũ nếu có
!pkill -f "python /content/ComfyUI/main.py" || true
!pkill -f "cloudflared" || true

# 3. Chạy ComfyUI với Legacy Frontend (Load tức thì, chống kẹt splash screen)
comfy_cmd = "python /content/ComfyUI/main.py --listen 127.0.0.1 --port 8188 --fp8_e4m3fn-text-enc --preview-method auto --enable-cors-header --front-end-version Comfy-Org/ComfyUI_legacy_frontend@latest"
comfy_proc = subprocess.Popen(comfy_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# 4. Khởi động Cloudflare Tunnel
tunnel_cmd = "cloudflared tunnel --url http://127.0.0.1:8188"
tunnel_proc = subprocess.Popen(tunnel_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("⏳ Đang khởi động ComfyUI và tạo đường dẫn công khai Cloudflare...")
tunnel_url = None
start_time = time.time()
while time.time() - start_time < 60:
    line = tunnel_proc.stdout.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(0.5)

if tunnel_url:
    print("\n=========================================================================")
    print(f"🔗 BẤM VÀO ĐÂY ĐỂ MỞ COMFYUI:  {tunnel_url}")
    print("=========================================================================\n")
    print("👉 HƯỚNG DẪN TẠO VIDEO:")
    print("  1. Kéo thả file Wan2_1_14B_GGUF_I2V_Workflow.json vào màn hình ComfyUI.")
    print("  2. Tải ảnh lên và bấm Queue Prompt để bắt đầu!")
else:
    print("⚠️ Đang theo dõi log:")

for line in comfy_proc.stdout:
    print(line, end='')